# Text classification with RNNs

In this lab, we will find the author of a sentence with recurrent networks, written in PyTorch and trained with Lightning. The corpus is the slides' one: the sentences of 2 works by each of 9 English-language authors, from 1821 to 1913 (Brontë, Melville, Wilde…).

1. From text to sequences of indices.
2. A recurrent classifier that ignores the padding.
3. Training a bidirectional GRU.
4. Without packing, the GRU reads the padding.
5. Simple or bidirectional GRU?
6. Pretrained, frozen GloVe embeddings.
7. A deeper model.
8. The authors the model mixes up.

This lab needs a GPU: check in `Runtime > Change runtime type` that the hardware accelerator is a GPU. Each training takes a few minutes there, against a quarter of an hour to over an hour on a CPU.

## Installation and downloads

PyTorch Lightning, the corpus, spaCy's English model and the GloVe vectors of part 6 (an 822 MB file): allow one to two minutes.

In [ ]:
!pip install -q lightning
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh classical-authors
!python -m spacy download en_core_web_sm
!wget -q https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip
!unzip -q -o glove.6B.zip glove.6B.300d.txt

## Importing the libraries

In [ ]:
import collections
import pathlib
from collections.abc import Iterable, Iterator

import lightning
import matplotlib.pyplot as plt
import numpy as np
import pandas
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import spacy
import torch
import tqdm.auto
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, TensorDataset, random_split

# Same random draws at each run: reproducible results
lightning.seed_everything(0)

## Preprocessing

The next cell prepares the data:

- each sentence, split by spaCy, is an example, and its label is its author, from 0 to 8;
- every named entity (a word or a group of words) is replaced with its type, first letter tripled to make it a new word (`PPPERSON`, `DDDATE`…): the model must recognize a style, not the characters' names;
- the corpus' directories have misspelled names (`Melwille`, `bronteCharlote`): `AUTHORS` gives each author's name;
- a sentence without any word (an opening bracket "[" split off by spaCy, punctuation alone) is skipped: empty once split into words, these 1,763 sentences would have a length of zero, which `pack_padded_sequence` rejects (part 2);
- 70% of the sentences are for training (`X_train_raw`, `y_train`), 30% for testing (`X_test_raw`, `y_test`), always the same ones.

It takes about two minutes.

In [ ]:
# Corpus directory → author's name
AUTHORS = {
  "bronteAnne": "Anne Brontë",
  "bronteCharlote": "Charlotte Brontë",
  "Cooper": "Cooper",
  "Emerson": "Emerson",
  "Hawthorne": "Hawthorne",
  "Melwille": "Melville",
  "Shaw": "Shaw",
  "Thoreau": "Thoreau",
  "wilde": "Wilde",
}

nlp = spacy.load(
  "en_core_web_sm",
  exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"),
)
nlp.enable_pipe("senter")


def replace_ners(tokens: Iterable[spacy.tokens.Token]) -> Iterator[str]:
  for token in tokens:
    if token.ent_iob_ == "O":
      yield token.text
    elif token.ent_iob_ == "B":
      prefix = token.ent_type_[0]
      yield f"{prefix}{prefix}{token.ent_type_}"


def get_data(directory: pathlib.Path) -> tuple[list[str], list[str]]:
  texts = []
  authors = []
  # <author>/<work>/<chapter>.txt
  paths = sorted(directory.glob("*/*/*.txt"))
  contents = (p.read_text(encoding="utf8").replace("\n", " ") for p in paths)
  for path, doc in tqdm.auto.tqdm(zip(paths, nlp.pipe(contents)), total=len(paths)):
    author = AUTHORS[path.parent.parent.name]
    for sentence in doc.sents:
      text = " ".join(replace_ners(sentence))
      # A sentence without any letter or digit would be empty once split into words
      if any(char.isalnum() for char in text):
        authors.append(author)
        texts.append(text)
  return texts, authors


texts, authors = get_data(pathlib.Path("classical-authors"))
label_encoder = sklearn.preprocessing.LabelEncoder()
y = torch.tensor(label_encoder.fit_transform(authors))
print(f"{len(texts)} sentences; labels 0 to 8: {', '.join(label_encoder.classes_)}")

X_train_raw, X_test_raw, y_train, y_test = sklearn.model_selection.train_test_split(
  texts, y, test_size=0.3, random_state=0
)
print(f"Training: {len(X_train_raw)} sentences, test: {len(X_test_raw)}")
for text, label in zip(X_train_raw[:3], y_train[:3]):
  print(f"{label_encoder.classes_[label]}: {text}")

## 1. From text to sequences of indices

The `Tokenizer` class of the next cell is the slides' one. It numbers the words of a corpus:

- a word is a sequence of characters between two spaces, once the text is lowercased and the punctuation replaced by spaces (except the apostrophe);
- `fit_on_texts(texts)` builds the vocabulary `vocabulary`, the list of the words in the order of their indices: `<pad>` (index 0, the padding), `<unk>` (index 1, any word out of the vocabulary), then the `max_tokens - 2` most frequent words of `texts`, from the most to the least frequent;
- `texts_to_sequences(texts)` replaces each word of each text with its index.

In [ ]:
class Tokenizer:
  """Map the most frequent words of a corpus to indices, from 2 up."""

  # Characters replaced with spaces before splitting into words
  filters = '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'

  def __init__(self, max_tokens: int) -> None:
    self.max_tokens = max_tokens
    self.table = str.maketrans(self.filters, " " * len(self.filters))
    self.vocabulary: list[str] = []
    self.word_index: dict[str, int] = {}

  def split(self, text: str) -> list[str]:
    return text.lower().translate(self.table).split()

  def fit_on_texts(self, texts: Iterable[str]) -> None:
    counts = collections.Counter(word for text in texts for word in self.split(text))
    # Indices 0 and 1 reserved for the padding and the unknown words
    most_common = [word for word, _ in counts.most_common(self.max_tokens - 2)]
    self.vocabulary = ["<pad>", "<unk>", *most_common]
    self.word_index = {word: index for index, word in enumerate(self.vocabulary)}

  def texts_to_sequences(self, texts: Iterable[str]) -> list[list[int]]:
    # A word out of the vocabulary becomes 1, <unk>
    return [
      [self.word_index.get(word, 1) for word in self.split(text)] for text in texts
    ]

The next cell turns the sentences into tensors of word indices, as in the slides: a `Tokenizer` of 20,000 words, learned from the training sentences only, then `vectorize`, which truncates each sentence's sequence of indices to the first 150 and completes the shorter ones with 0s at their end (the padding), up to the length of the longest one.

In [ ]:
tokenizer = Tokenizer(max_tokens=20_000)  # the 20,000 most frequent words
tokenizer.fit_on_texts(X_train_raw)  # vocabulary of the training set only
vocabulary = tokenizer.vocabulary


def vectorize(texts: list[str]) -> torch.Tensor:
  # At most 150 indices per sentence
  sequences = [
    torch.tensor(sequence[:150]) for sequence in tokenizer.texts_to_sequences(texts)
  ]
  # 0s at the end, up to the length of the longest sequence
  return pad_sequence(sequences, batch_first=True, padding_value=0)


X_train = vectorize(X_train_raw)
X_test = vectorize(X_test_raw)

print(tuple(X_train.shape))
print(X_train_raw[0])
print(X_train[0, :20])
print(" | ".join(vocabulary[:10]))

### How many words to keep?

The two choices of the previous cell, 150 indices per sentence and a vocabulary of 20,000 words, are decided from the data. Measure them:

- `lengths`, the number of words of each training sentence: the nonzero indices of its row of `X_train` (a sentence longer than 150 words counts as 150 there);
- the histogram of `lengths`, its median and its 99th percentile;
- the share of the training sentences that are truncated, those filling their 150 indices;
- the share of the words of the test sentences out of the vocabulary (index 1), and the one of the test sentences with at least one of them.

Which maximum length would you choose from this distribution? What do 150 indices lose, and what do they cost? Are the vocabulary's 20,000 words enough?

**Useful documentation**:

- https://numpy.org/doc/stable/reference/generated/numpy.percentile.html
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html

In [ ]:
# Your code here

#### Additional hints

The calls and the parameters to use:

- `(X_train != 0).sum(dim=1).numpy()`: the number of nonzero indices of each row, as a NumPy array
- `plt.hist(lengths, bins=75)`: 75 bars 2 words wide; `plt.yscale("log")` makes the long, rare sentences visible
- `np.median(lengths)`, `np.percentile(lengths, 99)`: half of the sentences are shorter than the median, 99 out of 100 than the 99th percentile
- `(lengths == 150).mean()`: the share of truncated sentences
- `(X_test == 1).sum() / (X_test != 0).sum()`: the share of unknown words; `(X_test == 1).any(dim=1)` tells for each sentence whether it has at least one

A skeleton to copy into the exercise cell, replacing each `...`:

```python
lengths = (X_train != ...).sum(dim=1).numpy()
plt.hist(lengths, bins=...)
plt.yscale("log")
plt.xlabel("Words per sentence")
plt.ylabel("Sentences")
plt.show()
print(f"Median: {np.median(...):.0f} words; 99th percentile: {np.percentile(...):.0f}")
print(f"Truncated sentences: {(lengths == ...).mean():.1%}")
print(f"Unknown words in test: {(X_test == ...).sum() / (X_test != ...).sum():.1%}")
print(
  f"Sentences with an unknown word: {(X_test == ...).any(dim=1).float().mean():.0%}"
)
```

#### Solution

In [ ]:
lengths = (X_train != 0).sum(dim=1).numpy()  # words of each sentence, at most 150
plt.hist(lengths, bins=75)
plt.yscale("log")  # long sentences are rare
plt.xlabel("Words per sentence")
plt.ylabel("Sentences")
plt.show()
print(
  f"Median: {np.median(lengths):.0f} words; "
  f"99th percentile: {np.percentile(lengths, 99):.0f}"
)
print(f"Truncated sentences: {(lengths == 150).mean():.1%}")
print(f"Unknown words in test: {(X_test == 1).sum() / (X_test != 0).sum():.1%}")
print(f"Sentences with an unknown word: {(X_test == 1).any(dim=1).float().mean():.0%}")

# A sentence has 17 words at the median, and 99% have fewer than 100: the 99th
# percentile, 100 indices, would do. 150 truncate only 0.2%, and the 50 extra indices
# cost neither accuracy nor much computation: the GRU skips the padding (part 2). The 20,000 words kept, out of the training sentences'
# 31,785, cover 98% of the test words: 1.8% are unknown (rare words, names spaCy didn't
# recognize). But 27% of the test sentences have at least one: a larger vocabulary
# would save a few, at the cost of a larger embedding table, for words too rare to be
# learned well in it.

## 2. A recurrent classifier that ignores the padding

The Lightning module `TextClassifier` of the next cell is the slides' one: it wraps a PyTorch model, `model`, which gives a score (a logit) to each of the 9 authors, trains it with the cross-entropy loss and the Adam optimizer, and measures its accuracy in training, validation and test, in its `step` method.

In [ ]:
class TextClassifier(lightning.LightningModule):
  """Lightning wrapper: predict the author of a padded sequence of words."""

  def __init__(self, model: nn.Module, learning_rate: float = 1e-3) -> None:
    super().__init__()
    self.model = model
    self.learning_rate = learning_rate
    # An example sentence, for the tensor shapes of the model's summary: 20
    # unknown words (1s), since a row of 0s would be empty
    self.example_input_array = torch.ones(1, 20, dtype=torch.long)

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    return self.model(sequences)

  def step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    sequences, authors = batch
    logits = self(sequences)
    loss = nn.functional.cross_entropy(logits, authors)
    accuracy = (logits.argmax(dim=1) == authors).float().mean()
    # Means over each epoch, weighted by the batches' sizes
    self.log_dict(
      {f"{stage}_loss": loss, f"{stage}_accuracy": accuracy},
      on_step=False,
      on_epoch=True,
      prog_bar=True,
    )
    return loss

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self.step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> None:
    self.step(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> None:
    self.step(batch, "test")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

### The recurrent model

The class `TextGRU` of the next cell is the slides' classifier, with more parameters for the models of the next parts:

- `embedding`: the embedding layer, a new 64-dimensional `nn.Embedding` layer if none is given (the GloVe vectors of part 6);
- `hidden_size`, `num_layers` and `bidirectional`: the number of units, of layers and of reading directions of the GRU (a simple GRU in part 5, two layers in part 7);
- `dropout`: the dropout rate on the embeddings, and between the GRU's layers when it has several.

The GRU doesn't read the padding (slide "Ignoring Padding in the RNN"): a sentence has 17 words at the median, followed by more than 130 zeros in `X_train`. `forward` passes it the sequences packed by `pack_padded_sequence`, with their real lengths, the number of nonzero indices of each row: its last hidden states are those of the last real words, in both reading directions.

In [ ]:
class TextGRU(nn.Module):
  """Embed the words, summarize them with a GRU, score each author."""

  def __init__(
    self,
    embedding: nn.Embedding | None = None,
    hidden_size: int = 64,
    num_layers: int = 1,
    bidirectional: bool = True,
    dropout: float = 0.0,
  ) -> None:
    super().__init__()
    if embedding is None:
      # A vector of 64 numbers per word, zero for the padding
      embedding = nn.Embedding(len(vocabulary), 64, padding_idx=0)
    self.embedding = embedding
    self.dropout = nn.Dropout(dropout)  # on the embeddings
    self.gru = nn.GRU(
      embedding.embedding_dim,
      hidden_size,
      num_layers=num_layers,
      bidirectional=bidirectional,
      dropout=dropout if num_layers > 1 else 0.0,  # between the layers
      batch_first=True,
    )
    directions = 2 if bidirectional else 1
    self.head = nn.Linear(hidden_size * directions, len(label_encoder.classes_))

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    embedded = self.dropout(self.embedding(sequences))
    # Real length of each sequence: the GRU skips the padding, and its last
    # hidden states are those of the last real words, in both directions
    lengths = (sequences != 0).sum(dim=1).cpu()
    packed = pack_padded_sequence(
      embedded, lengths, batch_first=True, enforce_sorted=False
    )
    _, h_n = self.gru(packed)
    if self.gru.bidirectional:
      # Last states of the forward and backward directions of the last layer
      summary = torch.cat([h_n[-2], h_n[-1]], dim=1)
    else:
      summary = h_n[-1]
    return self.head(summary)


print(TextGRU()(X_train[:4]).shape)

## 3. Training a bidirectional GRU

The function `train` of the next cell trains a model with Lightning, then evaluates it:

- the model is wrapped in a `TextClassifier`; the gradient's norm is clipped to 1 (`gradient_clip_val`), against exploding gradients, common in RNNs;
- batches of 256 sentences; 20% of the training sentences for validation, the same ones at each call;
- at most 20 epochs: `EarlyStopping` stops the training after 2 epochs without a decrease of the validation loss, and `ModelCheckpoint` keeps the weights of the best one, those of the model evaluated on the test set and of the model `model` after the call.

It shows the model's summary and its accuracy curves, and returns its test accuracy. The cell after it applies it to a bidirectional GRU with 64 units per reading direction.

In [ ]:
def train(model: nn.Module, name: str, max_epochs: int = 20) -> float:
  classifier = TextClassifier(model)
  print(ModelSummary(classifier, max_depth=-1))
  # The same 20% of the training sentences for validation at each call
  train_dataset, val_dataset = random_split(
    TensorDataset(X_train, y_train),
    [0.8, 0.2],
    generator=torch.Generator().manual_seed(0),
  )
  # Keeps the weights of the epoch with the lowest validation loss
  checkpoint = ModelCheckpoint(monitor="val_loss")
  trainer = lightning.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices=1,
    gradient_clip_val=1.0,  # clipping of the gradient's norm
    callbacks=[
      # tqdm's progress bar, rather than rich's, which Lightning picks when rich
      # is installed (Colab): it froze some trainings
      TQDMProgressBar(),
      EarlyStopping(monitor="val_loss", patience=2),
      checkpoint,
    ],
    logger=CSVLogger("logs", name=name),
    enable_model_summary=False,  # summary already shown
  )
  trainer.fit(
    classifier,
    DataLoader(train_dataset, batch_size=256, shuffle=True),
    DataLoader(val_dataset, batch_size=1024),
  )
  # Test with the weights of the best epoch, loaded back into model
  test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=1024)
  (result,) = trainer.test(classifier, test_loader, ckpt_path="best", verbose=False)
  accuracy = result["test_accuracy"]

  # Accuracies of each epoch, written by the CSVLogger on different lines for
  # training and validation: we group them
  metrics = (
    pandas.read_csv(pathlib.Path(trainer.logger.log_dir) / "metrics.csv")
    .groupby("epoch")
    .mean(numeric_only=True)
    .dropna(subset=["train_accuracy", "val_accuracy"])
  )
  best = metrics["val_loss"].idxmin()
  print(
    f"Best epoch: {best + 1}; training accuracy "
    f"{metrics['train_accuracy'][best]:.3f}, "
    f"validation {metrics['val_accuracy'][best]:.3f}, test {accuracy:.3f}"
  )
  plt.plot(metrics.index + 1, metrics["train_accuracy"], label="training")
  plt.plot(metrics.index + 1, metrics["val_accuracy"], label="validation")
  plt.axvline(best + 1, color="gray", linestyle="--", label="best epoch")
  plt.xlabel("Epoch")
  plt.ylabel("Accuracy")
  plt.legend()
  plt.show()
  return accuracy

In [ ]:
bigru = TextGRU()
bigru_accuracy = train(bigru, "bigru")

### First conclusions

Given the curves and the accuracies, what can be said of the quality of this training? Compare the test accuracy with chance (one in 9) and with a model that would always predict the most frequent author, Melville (18% of the sentences).

Your answer here

#### Solution

The model learns: 55% test accuracy, five times chance (11%) and three times the majority class (18%).

But it overfits: from the 3rd epoch, its training accuracy exceeds the validation one, and the gap widens, up to 71% against 56% at the best epoch, the 8th. The validation loss hardly decreases after the 5th epoch, then goes back up: `EarlyStopping` stops the training two epochs later, and `train` evaluates the best one's weights.

The model has 1.3 million parameters, 1.28 million of them in the embedding table, for 41,000 training sentences: it memorizes the words of the training sentences rather than each author's style.

## 4. Without packing, the GRU reads the padding

What does packing the sequences bring? Write `PaddedTextGRU`, a subclass of `TextGRU` whose `forward` method passes the GRU the sequences' embeddings as they are, padding included, without `pack_padded_sequence`. Train a `PaddedTextGRU()` with `train`, bidirectional like the model of part 3, and compare their test accuracies.

In `X_train`, a 17-word sentence is followed by 133 zeros. What does each reading direction of the GRU end its reading with? Which of the two does the padding hinder?

In [ ]:
# Your code here

### Additional hints

The calls and the parameters to use:

- `self.embedding`, `self.dropout`, `self.gru` and `self.head`: the layers of `TextGRU`, which `PaddedTextGRU` inherits; only `forward` changes
- `_, h_n = self.gru(embedded)`: the GRU reads the 150 steps of each row, padding included
- `torch.cat([h_n[-2], h_n[-1]], dim=1)`: the last states of both reading directions, as in `TextGRU`
- `bigru_accuracy` holds the test accuracy of the bidirectional GRU of part 3

A skeleton to copy into the exercise cell, replacing each `...`:

```python
class PaddedTextGRU(TextGRU):
  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    embedded = self.dropout(self.embedding(...))
    _, h_n = self.gru(...)
    return self.head(torch.cat([..., ...], dim=1))


padded_bigru = PaddedTextGRU()
padded_accuracy = train(..., "padded_bigru")
print(f"Packed sequences: {bigru_accuracy:.3f}, padding read: {...:.3f}")
```

### Solution

In [ ]:
class PaddedTextGRU(TextGRU):
  """TextGRU whose GRU reads the padding: no packing."""

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    embedded = self.dropout(self.embedding(sequences))
    # The 150 steps of each row, padding included
    _, h_n = self.gru(embedded)
    return self.head(torch.cat([h_n[-2], h_n[-1]], dim=1))


padded_bigru = PaddedTextGRU()
padded_accuracy = train(padded_bigru, "padded_bigru")
print(f"Packed sequences: {bigru_accuracy:.3f}, padding read: {padded_accuracy:.3f}")

Without packing, the bidirectional GRU does worse: 53% test accuracy, against 55% with packed sequences, and it learns more slowly (its validation accuracy is lower at each epoch).

Its forward direction reads the sentence, then its 133 zeros: its last hidden state is the one of the last zero, and what it has kept of the sentence must go through more than 130 steps without information. Its backward direction starts with the zeros, then reads the sentence backward: its last state is the one of the first word, and it still summarizes the whole sentence. So the padding mostly hinders the forward direction, and the model hardly relies on anything but the backward one: its accuracy is about that of a GRU with a single reading direction (part 5). Each epoch also lasts longer: the GRU computes the 150 steps of each sentence, instead of its words only.

## 5. Simple or bidirectional GRU?

Train with `train` a simple GRU with 64 units, which reads the sentences in one direction only, and compare it with the bidirectional GRU of part 3:

- the test accuracy;
- the number of parameters of the recurrent layer, and of the whole model;
- the duration of an epoch, which Lightning's progress bar shows during the training.

Is the second reading direction worth its cost here?

In [ ]:
# Your code here

### Additional hints

- `TextGRU(bidirectional=False)`: the same model, with a simple GRU
- the summary `train` shows gives the number of parameters of each layer; `sum(p.numel() for p in model.parameters())` the one of the whole model
- `bigru_accuracy` holds the test accuracy of the bidirectional GRU

A skeleton to copy into the exercise cell, replacing each `...`:

```python
gru = TextGRU(...)
gru_accuracy = train(..., "gru")
for model, accuracy in ((gru, ...), (bigru, bigru_accuracy)):
  parameters = sum(p.numel() for p in model.parameters())
  print(f"{model.gru}: test accuracy {accuracy:.3f}, {parameters} parameters")
```

### Solution

In [ ]:
gru = TextGRU(bidirectional=False)
gru_accuracy = train(gru, "gru")
for model, accuracy in ((gru, gru_accuracy), (bigru, bigru_accuracy)):
  parameters = sum(p.numel() for p in model.parameters())
  print(f"{model.gru}: test accuracy {accuracy:.3f}, {parameters} parameters")

The simple GRU does a little worse: 54% test accuracy, against 55% for the bidirectional GRU. This one-point gap is of the order of what changes from one run to another.

The second reading direction doubles the parameters of the recurrent layer (24,960 → 49,920), but that is only 2% of the model, mostly made of the embedding (1.28 million); an epoch lasts hardly longer. The gain is modest, but it costs almost nothing: the final state of a simple GRU mostly summarizes the end of the sentence, and the backward direction adds a summary of its beginning.

## 6. Pretrained, frozen GloVe embeddings

The embeddings of the previous models start from random vectors and are learned on our 41,000 training sentences. The [GloVe](https://nlp.stanford.edu/projects/glove/) vectors were learned on 6 billion words (Wikipedia and Gigaword): words with close meanings have close vectors there.

Read the file `glove.6B.300d.txt` into a dictionary `glove`, word → vector: each line is a word followed by its 300 coordinates, separated by spaces, and only the words of `vocabulary` matter to us. The cell after the solution turns it into an embedding layer.

How many of the vocabulary's 20,000 words does GloVe know? Which ones are missing, and why?

In [ ]:
# Your code here

### Additional hints

The calls and the parameters to use:

- `word, *coordinates = line.rstrip().split(" ")` separates the word from its coordinates; `np.array(coordinates, dtype="float32")` makes them a vector
- `set(vocabulary)`: a fast `word in ...` test over the file's 400,000 lines

A skeleton to copy into the exercise cell, replacing each `...`:

```python
words = set(vocabulary)
glove = {}
with open("glove.6B.300d.txt", encoding="utf8") as file:
  for line in file:
    word, *coordinates = line.rstrip().split(" ")
    if word in ...:
      glove[word] = np.array(coordinates, dtype="float32")

missing = [word for word in vocabulary if word not in ...]
print(f"{len(glove)} words out of {len(vocabulary)} in GloVe")
print("Missing:", ", ".join(missing[:30]))
```

### Solution

In [ ]:
# Only the vectors of the vocabulary's words are kept
words = set(vocabulary)
glove = {}
with open("glove.6B.300d.txt", encoding="utf8") as file:
  for line in file:
    word, *coordinates = line.rstrip().split(" ")
    if word in words:
      glove[word] = np.array(coordinates, dtype="float32")

missing = [word for word in vocabulary if word not in glove]
print(f"{len(glove)} words out of {len(vocabulary)} in GloVe")
print("Missing:", ", ".join(missing[:30]))

# 19,114 of the vocabulary's 20,000 words are in GloVe. Missing: the padding (<unk>
# itself has a vector in GloVe), the entity types (ppperson, dddate…), names spaCy
# didn't recognize (holgrave, pyncheon), the vocabulary of whaling (harpooneer,
# whalemen) and old spellings (d'ye, oftener): words that a 21st-century corpus
# (Wikipedia, the press) doesn't use.

### The GloVe embedding layer

The next cell builds `glove_embedding`, an `nn.Embedding` layer that gives each word of `vocabulary` its GloVe vector:

- `embedding_matrix`, of shape `(len(vocabulary), 300)`: its row `i` is the vector of the word `vocabulary[i]`, or stays zero if GloVe doesn't know this word;
- `nn.Embedding.from_pretrained` creates the layer from this matrix; `freeze=True` freezes it (its vectors don't change during training), and `padding_idx=0` keeps the padding's vector zero.

In [ ]:
embedding_matrix = np.zeros((len(vocabulary), 300), dtype="float32")  # 0 if missing
for i, word in enumerate(vocabulary):
  if word in glove:
    embedding_matrix[i] = glove[word]
glove_embedding = nn.Embedding.from_pretrained(
  torch.from_numpy(embedding_matrix),
  freeze=True,  # frozen table
  padding_idx=0,
)

### Training with GloVe

Train the same bidirectional GRU as in part 3, with `glove_embedding` as its embedding layer, and compare it with the first one:

- the gap between the training accuracy and the validation one;
- the test accuracy;
- the number of trainable parameters, which the model's summary gives.

In [ ]:
# Your code here

#### Additional hints

`TextGRU` takes the embedding layer through its first parameter, `embedding`. `bigru_accuracy` holds the test accuracy of the first model.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
glove_bigru = TextGRU(...)
glove_accuracy = train(..., "glove_bigru")
print(f"Learned embeddings: {bigru_accuracy:.3f}, frozen GloVe: {glove_accuracy:.3f}")
```

#### Solution

In [ ]:
glove_bigru = TextGRU(glove_embedding)
glove_accuracy = train(glove_bigru, "glove_bigru")
print(f"Learned embeddings: {bigru_accuracy:.3f}, frozen GloVe: {glove_accuracy:.3f}")

With frozen GloVe, the model has only 141,705 trainable parameters, against 1.3 million: the 6 million numbers of the GloVe table don't move.

It generalizes better: at the best epoch, the 9th, the gap between the training accuracy and the validation one is only 9 points (70% against 61%), instead of 15. Its validation loss goes lower, and the test accuracy rises from 55% to 59%. The GloVe vectors bring what our 41,000 sentences can't learn alone: words with close meanings have close vectors there, even when they are rare in the corpus.

Yet the gap ends up widening, to 73% against 60% two epochs later: the recurrent layer still memorizes. This is the role of the next model's dropout.

## 7. A deeper model

A last model, given: two layers of bidirectional GRU stacked (`num_layers=2`) on the GloVe embeddings. The second one reads the sequence of the first one's hidden states, word by word.

`dropout=0.5` sets to zero, at each training step, half of the input values of each recurrent layer (the embeddings, then the first layer's states), against overfitting.

In [ ]:
deep = TextGRU(glove_embedding, num_layers=2, dropout=0.5)
deep_accuracy = train(deep, "deep_bigru")
print(
  f"GloVe and one bidirectional GRU: {glove_accuracy:.3f}, two: {deep_accuracy:.3f}"
)

The deep model reaches 60% test accuracy, hardly more than the bidirectional GRU on GloVe (59%). Dropout has removed the overfitting: at the best epoch, the 13th, the training and validation accuracies stay close (60% and 61%), the former measured with dropout active. But it slows the learning down: the validation loss decreases for 13 epochs, against 9 without dropout.

## 8. The authors the model mixes up

The next cell shows the deep model's confusion matrix on the test sentences: an author's row gives how their sentences are spread among the predicted authors.

In [ ]:
deep.eval()  # without dropout
device = next(deep.parameters()).device
with torch.no_grad():
  y_pred = torch.cat(
    [deep(batch.to(device)).argmax(dim=1).cpu() for batch in X_test.split(1024)]
  )
fig, ax = plt.subplots(figsize=(8, 8))
sklearn.metrics.ConfusionMatrixDisplay.from_predictions(
  y_test,
  y_pred,
  display_labels=label_encoder.classes_,
  normalize="true",  # proportions of each row
  values_format=".0%",
  xticks_rotation=45,
  cmap="Blues",
  colorbar=False,
  ax=ax,
)
ax.set_xlabel("Predicted author")
ax.set_ylabel("True author")
plt.show()

### Reading the matrix

Which authors does the model recognize best? Which ones does it mix up, and what do they have in common? The corpus' works are listed in the [dataset's README](https://github.com/shuuchuu/datasets/tree/main/classical-authors).

Your answer here

#### Solution

In our run:

- Melville, Shaw and Wilde are the best recognized (65% of their sentences), then Cooper (61%): *Moby-Dick* has a vocabulary of its own (the sea, whaling), and Shaw's and Wilde's plays are made of short lines, which don't look like prose.
- But Shaw and Wilde get mixed up with each other (14% and 18% of their sentences): two Irish playwrights of the same period.
- Anne Brontë is taken for her sister Charlotte in 20% of the cases: two English novelists of the same period, whose novels are told in the first person, by a governess in *Agnes Grey* as in *Jane Eyre*. Charlotte, almost twice as represented, attracts the ambiguous sentences of every author: her column is the most loaded after the diagonal.
- So do Emerson and Thoreau (18% and 11%): two American essayists of transcendentalism, neighbors and friends.

A single sentence is weak evidence: many are short and commonplace ("Yes ."), and no model can attribute them. The deep model (60% test accuracy) hardly does better than the bidirectional GRU on GloVe (59%), and a logistic regression on a TF-IDF representation of the sentences reaches about 61% on the same split: on short sentences, the words used matter more than their order.